In [4]:
import pandas as pd
import json
import os

def group_poems_final(input_csv, output_csv, lines_per_group=4):
    if not os.path.exists(input_csv):
        raise FileNotFoundError(f"❌ Input file '{input_csv}' not found.")
        
    df = pd.read_csv(input_csv)
    # Sort to ensure correct poem order
    df = df.sort_values(['poem_id', 'Row_ID']).reset_index(drop=True)
    
    grouped_data = []
    
    for poem_id, poem_df in df.groupby('poem_id'):
        rows = poem_df.to_dict('records')
        
        for i in range(0, len(rows), lines_per_group):
            chunk = rows[i:i + lines_per_group]
            if not chunk:
                continue
            
            merged = merge_chunk_metadata(chunk, poem_id)
            grouped_data.append(merged)
    
    output_df = pd.DataFrame(grouped_data)
    # Ensure row_ids is stored as valid JSON string
    output_df['row_ids'] = output_df['row_ids'].apply(lambda x: json.dumps(x))
    
    output_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    
    print(f"✅ Processing Complete!")
    print(f"   Input Rows: {len(df)}")
    print(f"   Output Groups: {len(output_df)}")
    return output_df

def merge_chunk_metadata(chunk, poem_id):
    first_row = chunk[0]
    
    record = {
        'poem_id': poem_id,
        'row_ids': [r['Row_ID'] for r in chunk],
        'line_count': len(chunk),
        'Title_cleaned': first_row['Title_cleaned'],
        'Title_raw': first_row['Title_raw'],
        'category': first_row['category'],
        'meter': first_row['meter'],
        'qafiya': first_row['qafiya'],
        'ai_images_thumb': first_row['ai_images_thumb'],
    }
    
    # Combine Text
    record['Poem_line_cleaned'] = '\n'.join([r['Poem_line_cleaned'] for r in chunk])
    record['Poem_line_raw'] = '\n'.join([r['Poem_line_raw'] for r in chunk])
    
    # --- FIELDS WITH JSON OBJECTS (Deduplicate by 'name') ---
    # These are always lists of objects in your data
    object_fields = ['places', 'religion', 'animals', 'events', 'entities']
    for field in object_fields:
        record[field] = json.dumps(merge_unique_objects(chunk, field), ensure_ascii=False)
    
    # --- FIELDS WITH STRINGS OR LISTS (Deduplicate by value) ---
    # These can be "حب" OR ["حب", "شوق"]
    string_fields = ['sentiments', 'subjects']
    for field in string_fields:
        record[field] = json.dumps(merge_unique_flexible(chunk, field), ensure_ascii=False)
    
    # Summaries
    record['summary_combined'] = merge_unique_summaries(chunk)
    
    # AI Prompt
    record['ai_prompt'] = generate_enhanced_prompt(record)
    
    return record

def merge_unique_objects(chunk, field_name):
    """Handles list of objects: [{'name': 'X'}, {'name': 'Y'}]"""
    unique_items = []
    seen_names = set()
    
    for row in chunk:
        try:
            raw = row.get(field_name, '[]')
            if pd.isna(raw) or raw == '':
                continue
                
            # Parse JSON
            data = json.loads(raw) if isinstance(raw, str) else raw
            
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict):
                        name = item.get('name')
                        if name and name not in seen_names:
                            seen_names.add(name)
                            unique_items.append(item)
                    elif isinstance(item, str):
                        if item not in seen_names:
                            seen_names.add(item)
                            unique_items.append({"name": item}) # Normalize to object if needed
        except (json.JSONDecodeError, TypeError):
            continue
    return unique_items

def merge_unique_flexible(chunk, field_name):
    """
    Handles mixed formats: 
    1. Single string: "حب"
    2. JSON List: '["حب", "شوق"]'
    3. Python List: ['حب', 'شوق']
    """
    unique_items = []
    seen_items = set()
    
    for row in chunk:
        raw = row.get(field_name)
        
        # Skip empty/NaN
        if pd.isna(raw) or raw == '':
            continue
            
        data = None
        
        # Try parsing as JSON list first
        if isinstance(raw, str):
            raw = raw.strip()
            if raw.startswith('[') and raw.endswith(']'):
                try:
                    data = json.loads(raw)
                except json.JSONDecodeError:
                    data = None # Not valid JSON, treat as single string below
            
            # If it didn't look like a list or failed parsing, treat as single string
            if data is None:
                data = [raw] 
        else:
            data = raw if isinstance(raw, list) else [raw]
        
        # Process the list
        if isinstance(data, list):
            for item in data:
                if item and isinstance(item, str):
                    item = item.strip()
                    if item and item not in seen_items:
                        seen_items.add(item)
                        unique_items.append(item)
                        
    return unique_items

def merge_unique_summaries(chunk):
    unique_sentences = []
    seen_sentences = set()
    for row in chunk:
        summary = row.get('summary', '')
        if pd.notna(summary):
            summary = str(summary).strip()
            if summary and summary not in seen_sentences:
                seen_sentences.add(summary)
                unique_sentences.append(summary)
    return ' '.join(unique_sentences)

def generate_enhanced_prompt(record):
    try:
        sentiments = json.loads(record['sentiments'])
        animals = json.loads(record['animals'])
        places = json.loads(record['places'])
        
        animal_names = [a.get('name', str(a)) for a in animals] if animals else []
        place_names = [p.get('name', str(p)) for p in places] if places else []
        
        parts = [record['summary_combined']]
        if sentiments: parts.append(f"Mood: {', '.join(sentiments)}")
        if animal_names: parts.append(f"Animals: {', '.join(animal_names)}")
        if place_names: parts.append(f"Location: {', '.join(place_names)}")
        parts.append("Style: Arabic Nabati Poetry Visualization, Cinematic, High Detail")
        
        return ' | '.join(parts)
    except Exception as e:
        return record['summary_combined']

# ================= EXECUTION =================
if __name__ == "__main__":
    INPUT_FILE = '.\CSV\Diwan-Hamdan-WIP - Word_search.csv'
    OUTPUT_FILE = 'poems_grouped_4lines_FINAL.csv'
    
    try:
        df_result = group_poems_final(INPUT_FILE, OUTPUT_FILE, lines_per_group=4)
        
        print("\n🔍 Verification Output:")
        # Specifically check sentiments, subjects, and row_ids
        cols = ['row_ids', 'sentiments', 'subjects', 'animals', 'places']
        print(df_result[cols].head(3).to_string())
        
        # Check a specific row to ensure "حب" is captured
        first_sentiment = json.loads(df_result.iloc[0]['sentiments'])
        print(f"\n✅ First group sentiments: {first_sentiment}")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

✅ Processing Complete!
   Input Rows: 4207
   Output Groups: 1172

🔍 Verification Output:
                row_ids            sentiments                                                             subjects                                                              animals                                              places
0          [1, 2, 3, 4]  ["حب", "غضب", "حزن"]   ["الحب والغزل", "الشوق والحنين", "الغيرة والعتاب", "الألم والحزن"]                                                                   []                                                  []
1                [5, 6]         ["فخر", "حب"]  ["التراث النبطي", "الغيرة والعتاب", "الحب والغزل", "الشوق والحنين"]  [{"name": "الجمل", "category": "حيوان", "resolved_from": ["معرب"]}]                                                  []
2  [101, 102, 103, 104]                ["حب"]                                                   ["الشعر والإبداع"]                                                                   []  [{"resolved_from"